# 03 — Terrace 32


## Imports and repository paths


In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
ROOT = next((folder for folder in (WORKING_DIR, *WORKING_DIR.parents)
             if (folder / "utils/site_analysis.py").is_file()
             and (folder / "data/accelerometer").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository or its processing folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from utils import hvsr_tools as hv
from utils import plotting as hp
from utils import site_analysis as sa
from utils.analysis_figure import FIGURE_CONTROLS, make_analysis_figure


## Site and processing parameters


In [ ]:
SITE = sa.SITE_BY_KEY['03_terrace_32']
PARAMS = sa.BASE_PARAMS
FIGURE_DIR = ROOT / "main/02_hvsr_processing/figures" / SITE.key
DATA_DIR = ROOT / "main/02_hvsr_processing/data" / SITE.key
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
display(pd.Series({
    "field_label": SITE.label,
    "session_id": SITE.session_id,
    "processing_parameters": PARAMS.to_dict(),
}, name="Analysis setup"))


## Recording and HVSR processing


In [ ]:
analysis = sa.analyze_site(ROOT, SITE.key, PARAMS)
summary = sa.site_summary([analysis])
display(summary.round(3))
display(pd.Series(analysis.quality["metrics"], name="SESAME quality metrics"))


## Site analysis figure


In [ ]:
ANALYSIS_STYLE = hp.PaperStyle(width_in=11.5, font_size=8.0, tick_size=7.0,
                                   legend_size=7.0, dpi=300)
FOURIER_DATA = hv.component_fourier_data(analysis.record, analysis.result)
analysis_figure = make_analysis_figure(
    analysis.record,
    analysis.result,
    FOURIER_DATA.smoothed,
    FIGURE_CONTROLS,
    ANALYSIS_STYLE,
    show_windows=True,
    fourier_data=FOURIER_DATA,
)
analysis_figure.suptitle(SITE.figure_label, fontsize=ANALYSIS_STYLE.font_size)
figure_paths = hp.save_figure(
    analysis_figure,
    FIGURE_DIR / f"{SITE.key}_analysis",
    formats=("pdf", "svg", "png"),
    style=ANALYSIS_STYLE,
)
display(figure_paths)


## Data and diagnostics export


In [ ]:
summary.to_csv(DATA_DIR / f"{SITE.key}_summary.csv", index=False)
data_paths = hv.save_outputs(
    analysis.result,
    DATA_DIR,
    prefix=SITE.key,
    extra={"field_label": SITE.label, "quality": analysis.quality["metrics"]},
)
display(data_paths)
